In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import preprocessing
from sklearn.model_selection import train_test_split

## 訓練データの読み込みと変換
- ・store_nbr販売店舗の識別
- ・family:製品のタイプ
- ・sales:その日付で特定の製品タイプ(family)の売り上げ
- ・onpromotion:店舗で宣伝されていた製品タイプ(family)の総数

In [ ]:
df_train = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/train.csv')
df_train.head()

In [ ]:
df_train["date"]=pd.to_datetime(df_train.date)

## オイル価格データの読み込み
dcoilwtico:West TAxes Intermedicate:原油の値段データ?

In [ ]:
df_oil = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/oil.csv')
df_oil.head()

## 提出形式の確認

In [ ]:
df_sample_submission = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/sample_submission.csv')
df_sample_submission.head()

## 休日イベントデータの読み込み
- ・type:休日種類
- ・locale:イベント範囲
- ・locale_name:イベント地域
- ・descriptio:イベント名？
- ・transferred:イベントの振り替えが行われたかどうか

In [ ]:
df_holidays_events = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/holidays_events.csv')
df_holidays_events.head()

In [ ]:
df_holidays_events["date"] = pd.to_datetime(df_holidays_events.date)

## 店舗データの読み込み
- ・store_nbr:販売店舗の識別番号
- ・city:店舗所在都市
- ・state:店舗所在州
- ・type:店舗タイプ
- ・cluser:近隣の類似店舗数

In [ ]:
df_stores = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/stores.csv')
df_stores.head()

## テストデータの読み込み

In [ ]:
df_test = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/test.csv')
df_test.head()

In [ ]:
df_test["date"] = pd.to_datetime(df_test.date)

## 取引に関するデータの読み込み
- ・store_nbr:販売店舗の識別番号
- ・transacations:（その日に店舗で行われた）取引数？

In [ ]:
df_transactions = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/transactions.csv')
df_transactions.head()

## 売上データの確認

In [ ]:
df_train["sales"].describe()

In [ ]:
# グラフ表示
def visualization(data, kind, title, xlabel, ylabel, x=None, y=None):
    # kind: bar, line, box, scatter
    plt.figure(figsize=(16, 8))

    if kind == 'box':
        sns.boxplot(x=x, y=y, data=data)
    elif kind == 'scatter':
        sns.scatterplot(x=x, y=y, data=data)
    elif kind == 'lineplot':
        sns.lineplot(x=x, y=y, data=data)
    else:
        data.plot(kind=kind, color=None, title=title)

    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.show()

## 売り上げと取引に関するデータの可視化

In [ ]:
df_transactions["date"] = pd.to_datetime(df_transactions.date)

In [ ]:
sales_by_date = df_train.groupby('date')['sales'].sum()
visualization(sales_by_date, 'line', 'Daily Sales', 'Date', 'Sales')

transactions_by_date = df_transactions.groupby('date')['transactions'].sum()
visualization(transactions_by_date, 'line', 'Daily Transactions', 'Date', 'Transactions')

## 曜日ごとの売り上げ傾向

In [ ]:
df_tmp = df_train.pivot_table(index='date', columns=['store_nbr', 'family'], values='sales', aggfunc='sum')
df_tmp['total_sales'] = df_tmp.sum(axis=1)


df_total_sales = df_tmp.reset_index()
df_total_sales['day_of_week'] = df_total_sales['date'].dt.dayofweek

visualization(df_total_sales,\
              kind='box',\
              title='Total Daily Sales Distribution by Day of Week',\
              xlabel='Day of Week (0=Monday)',\
              ylabel='Total Sales',\
              x='day_of_week',\
              y='total_sales')

## 月ごとの売り上げ傾向

In [ ]:
df_tmp = df_train.pivot_table(index='date', columns=['store_nbr', 'family'], values='sales', aggfunc='sum')
df_tmp['total_sales'] = df_tmp.sum(axis=1)

df_total_sales = df_tmp.reset_index()
df_total_sales['month'] = df_total_sales['date'].dt.month

visualization(df_total_sales,\
              kind='box',\
              title='Total Daily Sales Distribution by Month',\
              xlabel='Month',\
              ylabel='Total Sales',\
              x='month',\
              y='total_sales')

In [ ]:
df_train['year'] = df_train.date.dt.year
df_train['month'] = df_train.date.dt.month
df_train['dayofmonth'] = df_train.date.dt.day

## オイルデータの傾向

In [ ]:
df_oil.isnull().sum()

In [ ]:
df_oil["date"] = pd.to_datetime(df_oil.date)
df_oil = df_oil.set_index("date").dcoilwtico.resample("D").sum().reset_index()
df_oil["dcoilwtico"] = df_oil["dcoilwtico"].replace(0, np.nan)
df_oil = df_oil.interpolate(method='linear')
df_oil.iloc[0] = df_oil.iloc[1] # 最初の値を補間
visualization(df_oil['dcoilwtico'],\
              kind='line',\
              title='Daily Oil Price',\
              xlabel='Date',\
              ylabel='Oil Price')

In [ ]:
tmp = pd.merge(
    df_train.groupby(["date", "store_nbr"]).sales.sum().reset_index(),
    df_transactions,
    how="left"
)
tmp = pd.merge(tmp, df_oil, how = "left")
print("Correlation")
print(tmp.drop(["store_nbr"], axis = 1).corr("spearman").dcoilwtico.loc[["sales", "transactions"]], "\n")

transactionとoilは相関がほぼないが、salesとoilの間は少し相関がある

In [ ]:
min_date = min(df_train['date'].min(), df_test['date'].min())
max_date = max(df_train['date'].max(), df_test['date'].max())
full_dates = pd.DataFrame({'date': pd.date_range(start=min_date, end=max_date)})
df_oil = pd.merge(full_dates, df_oil, how='left', on='date')

df_oil['dcoilwtico'] = df_oil['dcoilwtico'].interpolate(method='linear').bfill().ffill()

In [ ]:
df_oil = df_oil.set_index('date').dcoilwtico.interpolate(method='linear').to_frame()
df_train = pd.merge(df_train, df_oil, how='left', on='date')
df_test = pd.merge(df_test, df_oil, how='left', on='date')
print(df_train.isnull().sum())

In [ ]:
print(df_test.isnull().sum())

In [ ]:
# df_test['tmp_d'] = df_test.date.astype(str)
# df_test = pd.merge(df_test,df_oil,how='left',on='tmp_d')
# df_test.drop(columns='tmp_d',axis=1,inplace=True)
# df_oil.drop(columns='tmp_d',axis=1,inplace=True)

## 店ごとのプロモーションと売り上げの関係

In [ ]:
df_tmp = df_train.groupby('store_nbr')[['onpromotion', 'sales']].sum().reset_index()
visualization(df_tmp,
              kind='scatter',\
              title='Promotion and Sales Relationship',\
              xlabel='Total Promotions',\
              ylabel='Total Sales',\
              x='onpromotion',\
              y='sales')

## 地域ごとの平均売上

In [ ]:
df_train = pd.merge(df_train,df_stores,how='left',on='store_nbr')

df_tmp = df_train.groupby('city')['sales'].mean()
visualization(df_tmp,\
              kind='bar',\
              title='Average Sales by City',\
              xlabel='City',\
              ylabel='Average Sales',\
              x='city',\
              y='sales')

## 店舗タイプごとの平均売上

In [ ]:
df_tmp = df_train.groupby(['type']).sales.mean()
visualization(df_tmp,\
              kind='bar',\
              title='Average Sales by Type',\
              xlabel='Type',\
              ylabel='Average Sales',\
              x='type',\
              y='sales')

## 商品ごとの売り上げ

In [ ]:
df_tmp = df_train.groupby('family')['sales'].sum().sort_values(ascending=False)
visualization(df_tmp, 'bar', 'Sales by Family', 'Family', 'Sales')

In [ ]:
df_train.head()

In [ ]:
df_test.head()

In [ ]:
df_test['year'] = df_test.date.dt.year
df_test['month'] = df_test.date.dt.month
df_test['dayofmonth'] = df_test.date.dt.day
df_test = pd.merge(df_test,df_stores,how='left',on='store_nbr')

In [ ]:
df_train.drop(['id', 'cluster', 'state'], axis=1, inplace=True)
df_test.drop(['id', 'cluster', 'state'], axis=1, inplace=True)

## 前処理

In [ ]:
df_train.sales = np.log1p(df_train.sales)

In [ ]:
for i in ['family','city','type']:
    encoder = preprocessing.LabelEncoder()
    df_train[i] = encoder.fit_transform(df_train[i])
    df_test[i] =  encoder.transform(df_test[i]) 

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(df_train,df_train['sales'],test_size=0.05,shuffle=False)

In [ ]:
features = ['store_nbr','family','onpromotion','year','month','dayofmonth','dcoilwtico','city','type']

In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_log_error

xgb = XGBRegressor(tree_method='hist', n_estimators=500, early_stopping_rounds=10)
xgb.fit(X_train[features], y_train, eval_set=[(X_train[features], y_train), (X_val[features], y_val)], verbose=False)

predictions = xgb.predict(X_val[features])
predictions = np.maximum(predictions, 0)

predictions_exp = np.expm1(predictions)
y_val_exp = np.expm1(y_val)

msle = mean_squared_log_error(y_val_exp, predictions_exp)
print('MSLE:', msle)

In [ ]:
pred = xgb.predict(df_test[features])
pred = [max(0, p) for p in pred]
pred = np.expm1(pred)

submission = pd.read_csv('../input/store-sales-time-series-forecasting/sample_submission.csv', index_col='id')
submission['sales'] = pred

submission.to_csv('submission.csv')

print(submission.head())